# A0 W&B Run Plots

A0-only notebook backed by `wandb_metrics.py`. It loads and plots the three A0 config folders: `a0_hvg`, `a0_sparse16`, and `a0_aib`.


In [1]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
print("wandb_metrics:", wm.__file__)


wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py


## Choose A0 Runs To Load Or Download


In [6]:
import importlib
wm = importlib.reload(wm)

# Pick A0 config folder(s) under Topology_Task/configs, or use None / "None" / "all" for every run.
# Examples:
# CONFIG_FOLDERS_TO_DOWNLOAD = "a0_hvg"
# CONFIG_FOLDERS_TO_DOWNLOAD = ["a0_hvg", "a0_sparse16"]
# CONFIG_FOLDERS_TO_DOWNLOAD = "a0_hvg,a0_sparse16,a0_aib"
# CONFIG_FOLDERS_TO_DOWNLOAD = "all"
CONFIG_FOLDERS_TO_DOWNLOAD = ["a0_hvg", "a0_aib", "a0_sparse16",] # "a0_test_rerun"]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = False

# Useful when W&B has newer data than the local cache, especially for old scan_history fallback caches.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = True

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS_TO_DOWNLOAD)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)

Config folder filter:
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_hvg
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_aib
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_sparse16
Matched config run-name candidates: 54
RUN_NAME_REGEX = ^\s*(?:a0_hvg_00_baseline_s0|a0_hvg_00_baseline_s1|a0_hvg_00_baseline_s2|a0_hvg_01_eval_rho090_s0|a0_hvg_01_eval_rho090_s1|a0_hvg_01_eval_rho090_s2|a0_hvg_02_gate_final_map_s0|a0_hvg_02_gate_final_map_s1|a0_hvg_02_gate_final_map_s2|a0_hvg_03_gate_hierarchical_s0|a0_hvg_03_gate_hierarchical_s1|a0_hvg_03_gate_hierarchical_s2|a0_hvg_04_eval_local_rho090_s0|a0_hvg_04_eval_local_rho090_s1|a0_hvg_04_eval_local_rho090_s2|a0_aib_00_flat_local_t020_s0|a0_aib_00_flat_local_t020_s1|a0_aib_00_flat_local_t020_s2|a0_aib_01_flat_local_t010_s0|a0_aib_01_flat_local_t010_s1|a0_aib_01_flat_local_t010_s2|a0_aib_02_flat_local_t035_s0|a0_aib_02_flat_local_t035_s1|a0_aib_02_flat_local_t035_s2|a0_a

## Load Selected W&B Histories


In [7]:
data = wm.load_wandb_data()

runs_df = data.runs_df
history_df = data.history_df
#runs_df


Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 54 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
crashed     37
finished    17
History artifact setup: local_only=True, runs_df=54
[ 1/54] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 356 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib/runs/a0_aib_00_flat_local_t020_s0__MAPPO_bus14_T_0_0__I__1783010293_22044/history.parquet in 0.1s
[ 2/54] loading artifact cache: a0_aib_00_flat_local_t020_s1
    loaded 326 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib/runs/a0_aib_00_flat_local_t020_s1_

## A0 Heuristic vs Gate


Assuming baseline = `a0_hvg_00_baseline`.

| Run family | Displayed name | Eval heuristic | Intervention gate | Gate eval mode | Rho threshold | Difference vs `a0_hvg_00_baseline` |
|---|---|---|---|---|---:|---|
| `a0_hvg_00_baseline` | baseline | `none` | `false` | `-` | `0.90` | Baseline: plain A0 policy, no heuristic override and no learned gate |
| `a0_hvg_01_eval_rho090` | global rho heuristic | `rho_threshold` | `false` | `-` | `0.90` | Adds global rho-threshold heuristic at evaluation time |
| `a0_hvg_04_eval_local_rho090` | local rho heuristic | `local_rho_threshold` | `false` | `-` | `0.90` | Adds local/per-line rho-threshold heuristic at evaluation time |
| `a0_hvg_02_gate_final_map` | gate final-action MAP | `none` | `true` | `final_action_map` | `0.90` | Adds learned intervention gate with final-action MAP evaluation |
| `a0_hvg_03_gate_hierarchical` | gate hierarchical greedy | `none` | `true` | `hierarchical_greedy` | `0.90` | Adds learned intervention gate with hierarchical greedy evaluation |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


In [8]:
a0_hvg_result = wm.plot_a0_hvg_survival()
a0_hvg_result["fig"]


Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_survival_baseline_comparisons.html


## Intervention penalty (A0 Sparse16) (changing topology has a cost) 

Assuming baseline = `a0_sparse16_flat_p000`.

| Run family | Displayed name | Intervention gate | Intervention penalty | Gate eval mode | Difference vs `a0_sparse16_flat_p000` |
|---|---|---|---:|---|---|
| `a0_sparse16_flat_p000` | flat p0.000 | `false` | `0.000` | `-` | Baseline: flat action setup with no intervention penalty and no gate |
| `a0_sparse16_flat_p001` | flat p0.001 | `false` | `0.001` | `-` | Flat with small action/intervention penalty |
| `a0_sparse16_flat_p003` | flat p0.003 | `false` | `0.003` | `-` | Flat with medium action/intervention penalty |
| `a0_sparse16_flat_p010` | flat p0.010 | `false` | `0.010` | `-` | Flat with large action/intervention penalty |
| `a0_sparse16_gated_p000` | gated p0.000 | `true` | `0.000` | `final_action_map` | Adds intervention gate without intervention penalty |
| `a0_sparse16_gated_p001` | gated p0.001 | `true` | `0.001` | `final_action_map` | Adds intervention gate plus small intervention penalty |
| `a0_sparse16_gated_p003` | gated p0.003 | `true` | `0.003` | `final_action_map` | Adds intervention gate plus medium intervention penalty |
| `a0_sparse16_gated_p010` | gated p0.010 | `true` | `0.010` | `final_action_map` | Adds intervention gate plus large intervention penalty |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


In [9]:
a0_sparse16_result = wm.plot_a0_sparse16_survival()
a0_sparse16_result["fig"]


Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_survival_baseline_comparisons.html


## A0 Adaptive Intervention Budget


Assuming plot baseline = `a0_hvg_00_baseline` (plain A0 baseline).

| Run family | Displayed name | Adaptive budget | Cost mode | Target | Rho threshold | Intervention gate | Difference vs `a0_hvg_00_baseline` |
|---|---|---|---|---:|---:|---|---|
| `a0_hvg_00_baseline` | plain A0 baseline | `false` | `-` | `-` | `-` | `false` | Plain A0 baseline: no adaptive intervention budget and no gate |
| `a0_aib_00_flat_local_t020` | flat local target 0.20 | `true` | `local_safe` | `0.20` | `0.90` | `false` | Adds adaptive intervention budget with local-safe cost and target `0.20` |
| `a0_aib_01_flat_local_t010` | flat local target 0.10 | `true` | `local_safe` | `0.10` | `0.90` | `false` | Same local-safe budget mechanism with lower target `0.10` |
| `a0_aib_02_flat_local_t035` | flat local target 0.35 | `true` | `local_safe` | `0.35` | `0.90` | `false` | Same local-safe budget mechanism with higher target `0.35` |
| `a0_aib_03_gate_hgreedy_sep_local_t020` | gate h-greedy separate entropy target 0.20 | `true` | `local_safe` | `0.20` | `0.90` | `hierarchical_greedy` | Adds intervention gate with hierarchical greedy eval and separate gate entropy while keeping local-safe target `0.20` |
| `a0_aib_04_flat_nonidle_t020` | flat non-idle target 0.20 | `true` | `nonidle` | `0.20` | `0.90` | `false` | Uses non-idle cost instead of local-safe cost at target `0.20` |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


In [10]:
a0_aib_result = wm.plot_a0_aib_survival()
a0_aib_result["fig"]


Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_survival_vs_plain_a0_baseline.html
